> **Disclaimer:** This workshop is provided for educational and informational purposes only. The sample code and configurations are not intended for production use. Please review and adapt them according to your organization's security and compliance requirements. AWS service charges may apply for resources created during this workshop. Video content used in this workshop is sourced from publicly available materials and is attributed where applicable.

# Approach 1: Fused Embeddings (Baseline)

This notebook implements the simplest approach from the whitepaper: fusing visual, audio, and transcription embeddings into a single vector per segment.

**Formula:** `E_fused = normalize(w_v * E_visual + w_a * E_audio + w_t * E_transcript)`

**Default weights:** visual=0.8, audio=0.1, transcription=0.05

In [ ]:
import json
import numpy as np
import boto3
from botocore.config import Config

# Auto-load configuration from 01_setup_and_embedding.ipynb
with open("config.json") as f:
    config = json.load(f)

REGION = config["REGION"]
S3_BUCKET = config["S3_BUCKET"]
MODEL_ID_SYNC = config["MODEL_ID_SYNC"]
VECTOR_BUCKET = config["VECTOR_BUCKET"]
DATA_S3_PREFIX = config["DATA_S3_PREFIX"]
RESULTS_S3_PREFIX = config["RESULTS_S3_PREFIX"]

print(f"✅ Configuration loaded from config.json")
print(f"   S3 Bucket: {S3_BUCKET}")
print(f"   Vector Bucket: {VECTOR_BUCKET}")

bedrock = boto3.client("bedrock-runtime", region_name=REGION, config=Config(signature_version='v4'))
s3 = boto3.client("s3", region_name=REGION)
s3v = boto3.client("s3vectors", region_name=REGION)

def fmt_time(sec):
    """Convert seconds to mm:ss format"""
    m, s = divmod(sec, 60)
    return f"{int(m):02d}:{s:04.1f}"

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def embed_text(query):
    response = bedrock.invoke_model(
        modelId=MODEL_ID_SYNC,
        body=json.dumps({"inputType": "text", "text": {"inputText": query}})
    )
    result = json.loads(response["body"].read())
    return result["data"][0]["embedding"]

print("Setup complete")

## 1. Load Embeddings

In [ ]:
# Load embeddings from S3
def load_embeddings_from_s3(modality):
    s3_key = f"{DATA_S3_PREFIX}embeddings_{modality}.json"
    obj = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
    return json.loads(obj["Body"].read().decode("utf-8"))

visual = load_embeddings_from_s3("visual")
audio = load_embeddings_from_s3("audio")
transcription = load_embeddings_from_s3("transcription")

print(f"Visual segments: {len(visual)}")
print(f"Audio segments: {len(audio)}")
print(f"Transcription segments: {len(transcription)}")

## 2. Build Fused Index

Align segments by time range and compute fused embeddings.

In [ ]:
# Fusion weights (from whitepaper Section 2.2)
W_V, W_A, W_T = 0.8, 0.1, 0.05

def fuse_embeddings(v_emb, a_emb, t_emb, w_v=W_V, w_a=W_A, w_t=W_T):
    """Fuse 3 modality embeddings into a single vector (Equation 1 from whitepaper)"""
    v = np.array(v_emb)
    a = np.array(a_emb) if a_emb is not None else np.zeros_like(v)
    t = np.array(t_emb) if t_emb is not None else np.zeros_like(v)
    
    fused = w_v * v + w_a * a + w_t * t
    norm = np.linalg.norm(fused)
    if norm > 0:
        fused = fused / norm
    return fused.tolist()

# Build time-aligned index
# Create lookup by (startSec, endSec) for each modality
def build_time_lookup(embeddings):
    lookup = {}
    for e in embeddings:
        key = (round(e["startSec"], 1), round(e["endSec"], 1))
        lookup[key] = e["embedding"]
    return lookup

visual_lookup = build_time_lookup(visual)
audio_lookup = build_time_lookup(audio)
trans_lookup = build_time_lookup(transcription)

# Get all unique time segments from visual (primary modality)
fused_index = []
for seg in visual:
    key = (round(seg["startSec"], 1), round(seg["endSec"], 1))
    v_emb = seg["embedding"]
    a_emb = audio_lookup.get(key)
    t_emb = trans_lookup.get(key)
    
    fused = fuse_embeddings(v_emb, a_emb, t_emb)
    fused_index.append({
        "embedding": fused,
        "startSec": seg["startSec"],
        "endSec": seg["endSec"],
        "has_audio": a_emb is not None,
        "has_transcription": t_emb is not None
    })

print(f"Fused index: {len(fused_index)} segments")
print(f"  Dimension: {len(fused_index[0]['embedding'])}")
coverage = sum(1 for s in fused_index if s["has_audio"] and s["has_transcription"])
print(f"  Full 3-modality coverage: {coverage}/{len(fused_index)}")

## 2.5 Store Fused Embeddings in S3 Vectors

In [ ]:
FUSED_INDEX_NAME = "fused"

# Create fused index (skip if exists)
try:
    s3v.get_index(vectorBucketName=VECTOR_BUCKET, indexName=FUSED_INDEX_NAME)
    print(f"Index '{FUSED_INDEX_NAME}' already exists")
except s3v.exceptions.NotFoundException:
    s3v.create_index(
        vectorBucketName=VECTOR_BUCKET,
        indexName=FUSED_INDEX_NAME,
        dataType="float32",
        dimension=512,
        distanceMetric="cosine"
    )
    print(f"Created index '{FUSED_INDEX_NAME}' (512-dim, cosine)")

# Store fused embeddings
BATCH_SIZE = 50
total = 0
for i in range(0, len(fused_index), BATCH_SIZE):
    batch = fused_index[i:i+BATCH_SIZE]
    vectors = []
    for seg in batch:
        vectors.append({
            "key": f"fused_{seg['startSec']:.1f}_{seg['endSec']:.1f}",
            "data": {"float32": [float(x) for x in seg["embedding"]]},
            "metadata": {
                "startSec": seg["startSec"],
                "endSec": seg["endSec"]
            }
        })
    s3v.put_vectors(
        vectorBucketName=VECTOR_BUCKET,
        indexName=FUSED_INDEX_NAME,
        vectors=vectors
    )
    total += len(vectors)

print(f"Stored {total} fused vectors in S3 Vectors")

## 3. Search with Fused Embeddings

Test with queries spanning different modality intents.

In [ ]:
TEST_QUERIES = [
    {"query": "a player kicking the ball towards the goal", "intent": "visual"},
    {"query": "crowd cheering and celebrating loudly", "intent": "audio"},
    {"query": "the commentator describes the player's performance", "intent": "transcription"},
    {"query": "a goal being scored with the stadium erupting", "intent": "visual+audio"},
    {"query": "the announcer explaining a foul while the crowd boos", "intent": "mixed"},
    {"query": "players running on the soccer field", "intent": "visual"},
    {"query": "referee whistle blowing during the match", "intent": "audio"},
    {"query": "the commentator discusses the team's strategy", "intent": "transcription"},
]

def search_fused_s3v(query_text, top_k=5):
    """Search the fused index in S3 Vectors using ANN"""
    q_emb = embed_text(query_text)
    
    response = s3v.query_vectors(
        vectorBucketName=VECTOR_BUCKET,
        indexName=FUSED_INDEX_NAME,
        topK=top_k,
        queryVector={"float32": [float(x) for x in q_emb]},
        returnMetadata=True,
        returnDistance=True
    )
    
    results = []
    for v in response["vectors"]:
        results.append({
            "startSec": v["metadata"]["startSec"],
            "endSec": v["metadata"]["endSec"],
            "similarity": v["distance"]
        })
    return results

# Run searches using S3 Vectors
fused_results = {}
for item in TEST_QUERIES:
    query = item["query"]
    intent = item["intent"]
    print(f"\nQuery: \"{query}\" (intent: {intent})")
    print("-" * 60)
    
    results = search_fused_s3v(query)
    fused_results[query] = results
    
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{fmt_time(r['startSec'])} - {fmt_time(r['endSec'])}] distance: {r['similarity']:.4f}")

## 4. Limitations of Fused Embeddings

The fused approach uses the **same weights for every query**, regardless of intent:
- A visual query gets the same 80% visual bias as an audio query
- You can't tell which modality contributed to a match
- Changing weights requires re-computing all fused embeddings (irreversible)

In [ ]:
print("Limitation: When querying with the example weights,")
print(f"  Visual weight:        {W_V}")
print(f"  Audio weight:         {W_A}")
print(f"  Transcription weight: {W_T}")
print()
print("This means:")
print('  - "crowd cheering and celebrating loudly" still draws 80% from visual results.')
print('  - Cannot compare which modality matched, and tuning is difficult.')


In [ ]:
# Save results to S3
results_data = {
    "approach": "fused_embeddings",
    "weights": {"visual": W_V, "audio": W_A, "transcription": W_T},
    "results": {q: r for q, r in fused_results.items()}
}

s3_key = f"{RESULTS_S3_PREFIX}fused_results.json"
s3.put_object(
    Bucket=S3_BUCKET,
    Key=s3_key,
    Body=json.dumps(results_data, indent=2),
    ContentType="application/json"
)

print(f"Results saved to s3://{S3_BUCKET}/{s3_key}")

---

## Summary: Fused Embeddings Approach

### Architecture
```
[Video] -> Marengo 3.0 -> visual embedding (512-dim)
                        -> audio embedding (512-dim)       -> weighted sum -> fused embedding (512-dim) -> S3 Vectors
                        -> transcription embedding (512-dim)

[Search query] -> Marengo text embed -> cosine similarity search in fused index
```

### Core Formula
```
E_fused = normalize(0.8 x E_visual + 0.1 x E_audio + 0.05 x E_transcription)
```

### Purpose of This Notebook
- The key goal is to **demonstrate the limitations as a baseline**
- The simplest approach: merge 3 modality embeddings into one at storage time
- The weights used for merging (0.8/0.1/0.05) are **applied identically to all queries**

### What We Verify
| Query Intent | Expected | Actual (Fused Problem) |
|---|---|---|
| visual | Visual-based search -> appropriate results | O - Works well since 80% is visual |
| audio | Audio-based search -> cheering/whistle segments | X - Audio info is buried since 80% is visual |
| transcription | Commentary-based search | X - Transcription at 5% is nearly ignored |

### Limitations
1. **Irreversible**: Once merged, cannot recover original modality embeddings
2. **Query-agnostic**: Same weights for "crowd cheering" and "players running"
3. **Not debuggable**: Cannot tell which modality contributed to search results
4. **Weight change = full reprocessing**: Changing weights requires recomputing all fused embeddings

### Next Step
-> Notebook 03 explores what advantages keeping embeddings separate provides